# Phase 7 — vérification

Le contrôle reconstruit le fichier de performance, puis recalcule chaque mesure à partir des rendements mensuels. Il vérifie l'accord avec la phase 5 et avec la baisse du chemin publié en phase 6. Il ne fixe ni le nombre de titres, ni les dates.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import math

import numpy as np
import pandas as pd

from src.config import MONTHS_PER_YEAR
from src.performance.build_performance import (
    AGREEMENT,
    BENCHMARK_RETURNS_FILE,
    METRICS_OUTPUT,
    build_performance,
)
from src.portfolio_construction.build_portfolios import (
    METRICS_FILE,
    MONTHLY_RETURNS_FILE,
    STRATEGY_IDS,
)
from src.portfolio_monitoring.build_monitoring import PATH_FILE

written = build_performance()
performance = pd.read_csv(written["metrics"])
monthly = pd.read_csv(MONTHLY_RETURNS_FILE)
published = pd.read_csv(METRICS_FILE)
path = pd.read_csv(PATH_FILE)

assert list(performance["strategy"]) == list(STRATEGY_IDS)
benchmark = pd.read_csv(BENCHMARK_RETURNS_FILE)
assert performance["reference"].eq(benchmark["fund_name"].iloc[0]).all()
assert benchmark["month"].tolist() == (
    monthly.loc[monthly["strategy"].eq("equal_weight")]
    .sort_values("month", kind="mergesort")["month"]
    .tolist()
)
rebuilt = (1 + benchmark["nav_total_return"]) * (
    benchmark["chf_per_unit"] / benchmark["prior_chf_per_unit"]
) - 1
assert (rebuilt - benchmark["benchmark_return_chf"]).abs().max() <= AGREEMENT
assert performance["window_start"].nunique() == 1
assert performance["window_end"].nunique() == 1
assert performance["months"].nunique() == 1
assert written["metrics"] == METRICS_OUTPUT

for strategy in STRATEGY_IDS:
    row = performance.loc[performance["strategy"].eq(strategy)].iloc[0]
    official = published.loc[published["strategy"].eq(strategy)].iloc[0]
    block = monthly.loc[monthly["strategy"].eq(strategy)].sort_values("month", kind="mergesort")
    returns = block["portfolio_return"]
    excess = returns - block["risk_free_return"]
    growth = float(np.prod(1.0 + returns.to_numpy()))
    annualized = growth ** (MONTHS_PER_YEAR / len(returns)) - 1.0
    volatility = float(returns.std(ddof=1) * math.sqrt(MONTHS_PER_YEAR))
    sharpe = float(excess.mean() / excess.std(ddof=1) * math.sqrt(MONTHS_PER_YEAR))
    wealth = (1.0 + returns).cumprod()
    drawdown = wealth / wealth.cummax() - 1.0
    recomputed = {
        "annualized_return": annualized,
        "annualized_volatility": volatility,
        "sharpe_ratio": sharpe,
        "maximum_drawdown": float(drawdown.min()),
    }
    for name, value in recomputed.items():
        assert abs(float(row[name]) - value) <= AGREEMENT, name
        assert abs(float(official[name]) - value) <= AGREEMENT, name
    path_block = path.loc[path["strategy"].eq(strategy)].sort_values("month", kind="mergesort")
    assert abs(float(path_block["drawdown"].min()) - float(row["maximum_drawdown"])) <= AGREEMENT
    shortfall = np.minimum(excess.to_numpy(dtype=float), 0.0)
    below = int(np.count_nonzero(shortfall < 0.0))
    assert int(row["months_below_risk_free"]) == below
    deviation = math.sqrt(float(np.square(shortfall).sum()) / (len(excess) - 1))
    if deviation == 0.0:
        assert pd.isna(row["sortino_ratio"])
    else:
        sortino = float(excess.mean() / deviation * math.sqrt(MONTHS_PER_YEAR))
        assert abs(float(row["sortino_ratio"]) - sortino) <= AGREEMENT
    reference = benchmark.sort_values("month", kind="mergesort")["benchmark_return_chf"]
    gap = returns.to_numpy(dtype=float) - reference.to_numpy(dtype=float)
    active_std = float(pd.Series(gap).std(ddof=1))
    tracking = active_std * math.sqrt(MONTHS_PER_YEAR)
    assert abs(float(row["tracking_error"]) - tracking) <= AGREEMENT
    if active_std == 0.0:
        assert pd.isna(row["information_ratio"])
    else:
        information = float(gap.mean() / active_std * math.sqrt(MONTHS_PER_YEAR))
        assert abs(float(row["information_ratio"]) - information) <= AGREEMENT

from src.performance.dashboard import DASHBOARD_FILE, SECTION_TITLES, build_figures

equal = performance.iloc[0]
assert written["dashboard"] == DASHBOARD_FILE
dashboard_html = DASHBOARD_FILE.read_text(encoding="utf-8")
for title in SECTION_TITLES:
    assert title.split()[0] in dashboard_html
assert str(equal["window_end"]) in dashboard_html
for figure in build_figures()[:3]:
    for trace in figure.data:
        dates = pd.to_datetime(list(trace.x))
        assert dates.is_monotonic_increasing
        assert len(dates) > 1
print(
    f"Période : {equal['window_start']} – {equal['window_end']}. "
    f"Mois : {int(equal['months'])}."
)
print("Résultat : PASS")


Période : 2016-10 – 2026-09. Mois : 120.
Résultat : PASS
